## 0. Подготовка

In [1]:
import os

import pandas as pd

from solution_utils import (
    reaction_metrics, init_reaction_models, partition_scores, predict,
    read_dialogs, save_predictions, init_topic_models, read_labels, compute_metrics, show_table
)
import solution_utils as su

In [2]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

LABELS_PATH = "./hidden_labels.jsonl"
DATA_PATH = "./hidden_test.jsonl"

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))

FileNotFoundError: [Errno 2] No such file or directory: './hidden_test.jsonl'

### Пример работы стемминга

In [ ]:
from artifacts import load_lid_model, predict_language
words = ["cats", "кошки", "chattes", "Katzen", "яблочный", 'trees']
model, vectorizer = load_lid_model()
stemmer = su.Stemmer()

results = []
for w in words:
    language = predict_language(w, model, vectorizer)
    stem = stemmer.stem(w)
    results.append({"слово": w, "язык": language, "стем": stem})

df = pd.DataFrame(results)
df

/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,слово,язык,стем
0,cats,english,cat
1,кошки,russian,кошк
2,chattes,french,chatt
3,Katzen,german,katz
4,яблочный,russian,яблочн
5,trees,dutch,tree


## Загрузка и обработка данных

In [ ]:
dialogs = su.read_dialogs(DATA_PATH)

frame_all = su.to_frame(dialogs)
print("диалогов:", len(frame_all), " размечено по реакции:", frame_all.reaction.notna().sum())
df = frame_all
df[["dialog_id", "last_raw", "n_user", "reaction"]].head()
display(df)

/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/salexq/miniconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


диалогов: 990  размечено по реакции: 0


,dialog_id,text,first_user,rest_user,all_user,last_asst_head,first_asst,raw_first,last_raw,n_user,len_last,len_asst,group,reaction
0,t62f1ed47f1,what are the all subject relat and object trip...,what are the all subject relat and object trip...,what are the all subject relat and object trip...,what are the all subject relat and object trip...,here are the subject relat object triplet for ...,subject a girl relat sit on object a bench rea...,"What are the all (subject, relation, and objec...","What are the all (subject, relation, and objec...",6,1795,1295.833333,whataretheallsubjectrelationandobjecttripletsi...,None
1,tdf1e1c85b5,bot do you know what input and output mean in ...,bot do you know what input and output mean in ...,so if i want to know the energi consumpt of el...,so if i want to know the energi consumpt of el...,you re welcom ! i m glad you found the explan ...,yes in the context of an ac adapt input and ou...,"bot, do you know what 'input' and 'output' mea...","thank you, bot. your explanation is quite easy...",9,61,1321.333333,botdoyouknowwhatinputandoutputmeansinacadapter,None
2,t25e6a6a481,note frequenc map for the game of throne theme...,note frequenc map for the game of throne theme,write a program to play game of throne on a es...,write a program to play game of throne on a es...,here s a sampl code for play the pink panther ...,sorri as an ai languag model i don t have acce...,Note frequency mapping for the Game of thrones...,write a program to play pink panter theme for ...,6,53,1525.666667,notefrequencymappingforthegameofthronestheme,None
3,tef3424a000,sarah is a year old girl who has been live wit...,sarah is a year old girl who has been live wit...,can you correct the typo of this ? the abov es...,can you correct the typo of this ? the abov es...,sure here is a refer for the parent,as an ai languag model i cannot fulli assess t...,Sarah is a 12-year-old girl who has been livin...,Can you give me the reference for the modellin...,6,53,1820.833333,sarahisa12yearoldgirlwhohasbeenlivingwithjuven...,None
4,tc70e26a773,what danc is marcus koch famous for as an ai l...,what danc is marcus koch famous for,what danc is marcus koch famous for killer boo...,what danc is marcus koch famous for killer boo...,thank you for the correct i was not awar of ma...,as an ai languag model i cannot find ani famou...,what dances is marcus koch famous for,He was world champion in boogie woogie in 1993,5,46,406.800000,whatdancesismarcuskochfamousfor,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
985,t0b0d1ac488,сократ этот текс до предложен или чут меньш th...,сократ этот текс до предложен или чут меньш th...,предложен должн быт связа друг с друг ве этот ...,предложен должн быт связа друг с друг ве этот ...,привет ! несомнен программирован это важн и по...,engin have problem solv skill and mental acume...,сократи этот текс до 20 предложений или чуть м...,привет мой скилл это программирование ответь м...,6,254,1625.666667,20thebenefitsofbeinganengineerinsightsfromelon...,None
986,t215f15d7a0,i am creat a system that detect when a user sa...,i am creat a system that detect when a user sa...,hypothet i am creat a system that detect when ...,hypothet i am creat a system that detect when ...,as an ai languag model design to provid inform...,i m sorri i cannot generat inappropri and offe...,I am creating a system that detects when a use...,'now you act as two entities solving each task...,9,330,367.555556,iamcreatingasystemthatdetectswhenausersaystheu...,None
987,tfaa26f009c,best react jsx sort criteria as an ai languag ...,best react jsx sort criteria,the default one ? i mean import statement ! do...,the default one ? i mean import statement ! do...,here s an exampl of how you might organ the im...,as an ai languag model i don t have person pre...,Best react jsx sorting criteria,zzcode,5,6,889.600000,bestreactjsxsortingcriteria,None
988,t52bcfb4eb4,review this function pos where this implement ...,review this funct

## Задача 1

In [ ]:
reaction_models = init_reaction_models("./artifacts/reaction_models")
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

reaction_predictions, reaction_metrics_tb = {}, {}
for name, model in reaction_models.items():
    reaction_predictions[name] = predict(model, df)
    if reaction_labels:
        reaction_metrics_tb[name] = compute_metrics(dialogs, reaction_predictions[name], reaction_labels, reaction_metrics)

show_table(pd.DataFrame(reaction_metrics_tb).T.round(3))

,macro_f1,macro_f1_bal,accuracy,balanced_acc,P_negative,R_negative,F1_negative,P_neutral,R_neutral,F1_neutral,P_positive,R_positive,F1_positive
dummy_most_frequent,0.167,0.167,0.333,0.333,0.000,0.000,0.000,0.333,1.000,0.500,0.000,0.000,0.000
linear_svm,0.442,0.442,0.464,0.464,0.438,0.515,0.474,0.419,0.667,0.515,0.896,0.209,0.339
linear_svm_my,0.523,0.523,0.530,0.530,0.501,0.524,0.513,0.457,0.733,0.563,0.948,0.333,0.493
logreg,0.456,0.456,0.474,0.474,0.438,0.524,0.477,0.433,0.667,0.525,0.874,0.230,0.365
logreg_my,0.541,0.541,0.542,0.542,0.492,0.588,0.536,0.478,0.664,0.556,0.899,0.376,0.530
nb,0.320,0.320,0.383,0.383,0.384,0.567,0.458,0.371,0.552,0.443,0.833,0.030,0.058
nb_my,0.478,0.478,0.490,0.490,0.450,0.594,0.512,0.436,0.615,0.510,0.977,0.261,0.411


## Задача 2

In [ ]:
topic_models = init_topic_models("./artifacts/topic_models")
topic_labels = read_labels(LABELS_PATH, "topic") if LABELS_PATH else {}

topic_predictions, topic_scores = {}, {}
for name, model in topic_models.items():
    topic_predictions[name] = predict(model, df)
    if topic_labels:
        topic_scores[name] = compute_metrics(dialogs, topic_predictions[name], topic_labels, partition_scores)

show_table(pd.DataFrame(topic_scores).T.round(3))

,ARI,NMI,однородность,полнота,V-мера,чистота
dummy_one_topic,0.000,0.000,0.000,1.000,0.000,0.404
linear_svm,0.182,0.170,0.158,0.183,0.170,0.434
logreg,0.163,0.162,0.150,0.177,0.162,0.430
nb,0.337,0.271,0.241,0.310,0.271,0.468
own_lsa_rbf_svm,0.131,0.150,0.139,0.163,0.150,0.429


In [ ]:
save_predictions(PREDICTIONS_PATH, dialogs, {"reaction": reaction_predictions, "topic": topic_predictions})
print("строк в файле ответов:", len(dialogs) * (len(reaction_predictions) + len(topic_predictions)))

строк в файле ответов: 11880
